# Experiment 5: KNN Classification


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder


In [ ]:
# Load the dataset
df = pd.read_csv("fill it with the path of breast-cancer.csv")

print("Dataset loaded successfully!")
print(f"Dataset shape: {df.shape}")
print("\nFirst 5 rows:")
display(df.head())

In [ ]:
# Check dataset info
print("Dataset Info:")
df.info()

print("\nDataset Description:")
display(df.describe())

print("\nColumn Names:")
print(df.columns.tolist())

In [ ]:
# Drop unnecessary columns (like ID if present)
if 'id' in df.columns:
    df = df.drop('id', axis=1)
if 'Unnamed: 32' in df.columns:
    df = df.drop('Unnamed: 32', axis=1)



In [ ]:

# Visualize target distribution
plt.figure(figsize=(8,5))
df['diagnosis'].value_counts().plot(kind='bar', color=['blue', 'red'])
plt.title('Distribution of Diagnosis (M=Malignant, B=Benign)')
plt.xlabel('Diagnosis')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Encode target variable (M=Malignant=1, B=Benign=0)
label_encoder = LabelEncoder()
df['diagnosis'] = label_encoder.fit_transform(df['diagnosis'])

print("\nEncoded target variable:")
print(df['diagnosis'].value_counts())

In [ ]:
# Select required features
features = ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean']

X = df[features]
y = df['diagnosis']



In [ ]:
# Standardize features (mean=0, std=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features scaled successfully!")
print(f"Scaled features shape: {X_scaled.shape}")

# Convert to DataFrame for better visualization
X_scaled_df = pd.DataFrame(X_scaled, columns=features)
print("\nScaled Features (first 5 rows):")
display(X_scaled_df.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)



In [ ]:
# Initialize and train KNN classifier with K=5
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)



In [ ]:
# Predict on test set
y_pred = knn_model.predict(X_test)



In [ ]:
# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Model Performance Metrics (K=5):")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malignant']))

In [ ]:
# Find misclassified samples
misclassified_idx = np.where(y_test.values != y_pred)[0]

print(f"Number of misclassified cases: {len(misclassified_idx)}")
print(f"Total test samples: {len(y_test)}")
print(f"Misclassification rate: {len(misclassified_idx)/len(y_test)*100:.2f}%")

if len(misclassified_idx) > 0:
    print("\nFirst 5 misclassified cases:")
    for i in misclassified_idx[:5]:
        print(f"Index {i}: Actual={y_test.values[i]}, Predicted={y_pred[i]}")

In [ ]:
# Test different K values
k_values = range(1, 31)
accuracy_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred_k = knn.predict(X_test)
    acc = accuracy_score(y_test, y_pred_k)
    accuracy_scores.append(acc)

print("Testing different K values completed!")

# Find best K
best_k = k_values[np.argmax(accuracy_scores)]
best_accuracy = max(accuracy_scores)

print(f"\nBest K value: {best_k}")
print(f"Best accuracy: {best_accuracy:.4f}")

In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Benign', 'Malignant'], 
            yticklabels=['Benign', 'Malignant'])
plt.title('Confusion Matrix (K=5)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
# Plot accuracy vs K
plt.figure(figsize=(12, 6))
plt.plot(k_values, accuracy_scores, marker='o', linestyle='-', color='blue')
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best K={best_k}')
plt.xlabel('K Value (Number of Neighbors)')
plt.ylabel('Accuracy')
plt.title('KNN: Accuracy vs K Value')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
print(f"Optimal K value: {best_k} with accuracy: {best_accuracy:.4f}")

In [ ]:
# Train KNN with only 2 features for visualization
X_2d = df[['radius_mean', 'texture_mean']].values
X_2d_scaled = StandardScaler().fit_transform(X_2d)

# Split data
X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d_scaled, y, test_size=0.2, random_state=42
)

# Train model
knn_2d = KNeighborsClassifier(n_neighbors=best_k)
knn_2d.fit(X_train_2d, y_train_2d)

# Create mesh grid
h = 0.02  # step size
x_min, x_max = X_2d_scaled[:, 0].min() - 1, X_2d_scaled[:, 0].max() + 1
y_min, y_max = X_2d_scaled[:, 1].min() - 1, X_2d_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Predict on mesh
Z = knn_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot decision boundary
plt.figure(figsize=(10, 8))
plt.contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
plt.scatter(X_2d_scaled[y==0, 0], X_2d_scaled[y==0, 1], 
           c='blue', label='Benign', edgecolors='k', s=50)
plt.scatter(X_2d_scaled[y==1, 0], X_2d_scaled[y==1, 1], 
           c='red', label='Malignant', edgecolors='k', s=50)
plt.xlabel('Radius Mean (Scaled)')
plt.ylabel('Texture Mean (Scaled)')
plt.title(f'KNN Decision Boundary (K={best_k})')
plt.legend()
plt.show()

print("Decision boundary visualization completed!")

In [ ]:
# Train final model with best K
final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train, y_train)

# Predictions
final_pred = final_knn.predict(X_test)

# Final evaluation
final_accuracy = accuracy_score(y_test, final_pred)
final_precision = precision_score(y_test, final_pred)
final_recall = recall_score(y_test, final_pred)
final_f1 = f1_score(y_test, final_pred)

print(f"Final Model Performance (K={best_k}):")
print(f"Accuracy: {final_accuracy:.4f}")
print(f"Precision: {final_precision:.4f}")
print(f"Recall: {final_recall:.4f}")
print(f"F1 Score: {final_f1:.4f}")

print("KNN Classification Experiment Completed!")

# SCENARIO 2: Decision Tree Classifier
